# param-group-dict-list — worked example 2: no-decay biases split into a third group

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `param-group-dict-list`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Weight matrices (`p.dim() > 1`) usually get weight decay, while biases and norm scales (`p.dim() <= 1`) get none. Splitting the encoder by rank yields two groups, plus a head group, all distinguished by per-group hyperparameters.

## Worked solution

We implement `make_three_groups(encoder, head, encoder_lr, head_lr, weight_decay)`. We materialize encoder and head params, then partition the encoder by `p.dim() > 1` into decay-eligible weight matrices and no-decay vectors. We return three dicts in order: encoder-decay (with `weight_decay`), encoder-no-decay (with `weight_decay=0.0`), and head (with its own lr). The partition guarantees every encoder param lands in exactly one group with none dropped or duplicated. We build small modules, construct the groups, and print the per-group sizes and weight-decay values to confirm the rank-based split.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(1)

def make_three_groups(encoder, head, encoder_lr, head_lr, weight_decay):
    enc = list(encoder.parameters())
    decay = [p for p in enc if p.dim() > 1]
    no_decay = [p for p in enc if p.dim() <= 1]
    return [
        {'params': decay, 'lr': encoder_lr, 'weight_decay': weight_decay},
        {'params': no_decay, 'lr': encoder_lr, 'weight_decay': 0.0},
        {'params': list(head.parameters()), 'lr': head_lr, 'weight_decay': weight_decay},
    ]

encoder = nn.Linear(6, 6)   # weight (2-D) + bias (1-D)
head = nn.Linear(6, 3)
groups = make_three_groups(encoder, head, 1e-4, 1e-2, 0.01)
print('group sizes:', [len(g['params']) for g in groups])
print('weight_decays:', [g['weight_decay'] for g in groups])